# Fast MVTec AD preprocessing audit

Run this separately from `01_dataset_exploration.ipynb`. It does **not** modify or replace that notebook or its saved outputs.

Purpose: collect enough evidence to design preprocessing. It checks file structure and image/mask metadata across the full dataset, but reads pixel values only from small, deterministic samples to avoid slow Google Drive I/O.

Outputs are saved in `CS663_Shared/reports/preprocessing_audit_fast/`.

In [ ]:
import os
import subprocess
from pathlib import Path
import csv, json, random
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter, defaultdict

try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = Path('/content/CS-663')
    if REPO.exists():
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
    else:
        subprocess.run(['git', 'clone', 'https://github.com/pranitas-cloud/CS-663.git', str(REPO)], check=True)
except ImportError:
    REPO = Path.cwd()
    if not (REPO / 'src').is_dir():
        raise RuntimeError('Run this notebook from the CS-663 repository root.')

DATA_ROOT_OVERRIDE = os.environ.get('MVTEC_DATA_ROOT')
DATA_ROOT = Path(DATA_ROOT_OVERRIDE).expanduser() if DATA_ROOT_OVERRIDE else REPO / 'data' / 'mvtec_anomaly_detection'
if not DATA_ROOT.is_absolute():
    DATA_ROOT = REPO / DATA_ROOT
OUT_DIR = REPO / 'results' / 'preprocessing_audit_fast'
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
PIXEL_SAMPLES_PER_GROUP = 5
MASK_SAMPLES_PER_DEFECT = 5

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset root not found: {DATA_ROOT}. Set MVTEC_DATA_ROOT.')
categories = sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir() and (p/'train'/'good').is_dir())
print(f'Found {len(categories)} categories:', categories)

## 1. Full structural and image metadata scan
Opens every image only far enough to read its header (dimensions, mode, format). It does not decode all pixel arrays.

In [ ]:
image_rows = []
mask_rows = []
missing_masks = []
def read_header(path):
    with Image.open(path) as im:
        return {'width': im.width, 'height': im.height, 'mode': im.mode,
                'format': im.format, 'bands': str(im.getbands()),
                'aspect_ratio': round(im.width / im.height, 6) if im.height else None}

for category in categories:
    base = DATA_ROOT / category
    for split_dir in [base/'train'/'good', base/'test']:
        if not split_dir.exists():
            continue
        split = 'train' if split_dir.parent.name == 'train' else 'test'
        for p in sorted(split_dir.rglob('*.png')):
            defect = 'good' if split == 'train' else p.parent.name
            try:
                meta = read_header(p)
                image_rows.append({'category':category, 'split':split, 'defect_type':defect,
                                   'path':str(p), **meta})
            except Exception as e:
                image_rows.append({'category':category, 'split':split, 'defect_type':defect,
                                   'path':str(p), 'read_error':repr(e)})
            if split == 'test' and defect != 'good':
                mp = base/'ground_truth'/defect/f'{p.stem}_mask.png'
                if not mp.exists():
                    missing_masks.append({'category':category,'defect_type':defect,
                                          'image_path':str(p),'expected_mask_path':str(mp)})

    gt = base/'ground_truth'
    if gt.exists():
        for mp in sorted(gt.rglob('*.png')):
            try:
                meta = read_header(mp)
                mask_rows.append({'category':category,'defect_type':mp.parent.name,
                                  'path':str(mp),**meta})
            except Exception as e:
                mask_rows.append({'category':category,'defect_type':mp.parent.name,
                                  'path':str(mp),'read_error':repr(e)})

images_df = pd.DataFrame(image_rows)
masks_df = pd.DataFrame(mask_rows)
missing_df = pd.DataFrame(missing_masks)
images_df.to_csv(OUT_DIR/'all_image_metadata.csv', index=False)
masks_df.to_csv(OUT_DIR/'all_mask_metadata.csv', index=False)
missing_df.to_csv(OUT_DIR/'missing_mask_pairs.csv', index=False)
print('Images scanned:', len(images_df), '| masks scanned:', len(masks_df), '| missing mask pairs:', len(missing_masks))
display(images_df.groupby(['split','mode','width','height'], dropna=False).size().rename('count').reset_index().sort_values('count', ascending=False).head(30))
display(masks_df.groupby(['mode','width','height'], dropna=False).size().rename('count').reset_index().sort_values('count', ascending=False).head(30))


## 2. Pixel statistics from deterministic samples
Reads only up to five images per category and split. This is for colour-channel, dtype, range, and intensity evidence—not an exhaustive pixel census.

In [ ]:
rng = random.Random(SEED)
pixel_rows = []
for category in categories:
    base = DATA_ROOT/category
    groups = {'train_good': sorted((base/'train'/'good').glob('*.png'))}
    test_root = base/'test'
    if test_root.exists():
        for defect_dir in sorted(p for p in test_root.iterdir() if p.is_dir()):
            groups[f'test_{defect_dir.name}'] = sorted(defect_dir.glob('*.png'))
    for group, paths in groups.items():
        if not paths: continue
        chosen = rng.sample(paths, min(PIXEL_SAMPLES_PER_GROUP, len(paths)))
        for p in chosen:
            try:
                with Image.open(p) as im:
                    arr = np.asarray(im)
                row = {'category':category,'group':group,'file':p.name,
                       'shape':str(arr.shape),'dtype':str(arr.dtype),
                       'min':float(np.min(arr)),'max':float(np.max(arr)),
                       'mean':float(np.mean(arr)),'std':float(np.std(arr))}
                if arr.ndim == 3:
                    for c in range(arr.shape[2]):
                        row[f'ch{c}_mean'] = float(np.mean(arr[:,:,c]))
                        row[f'ch{c}_std'] = float(np.std(arr[:,:,c]))
                pixel_rows.append(row)
            except Exception as e:
                pixel_rows.append({'category':category,'group':group,'file':p.name,'read_error':repr(e)})

pixels_df = pd.DataFrame(pixel_rows)
pixels_df.to_csv(OUT_DIR/'sampled_image_pixel_statistics.csv', index=False)
display(pixels_df.head(20))
print('Pixel arrays decoded:', len(pixels_df))


## 3. Ground-truth mask value and coverage samples
All mask headers were scanned above. Here, decode only up to five masks per category/defect type to inspect unique values and foreground coverage.

In [ ]:
mask_pixel_rows = []
if len(masks_df):
    for (category, defect), group in masks_df.groupby(['category','defect_type']):
        paths = sorted(Path(p) for p in group['path'].tolist())
        chosen = rng.sample(paths, min(MASK_SAMPLES_PER_DEFECT, len(paths)))
        for p in chosen:
            try:
                with Image.open(p) as im:
                    arr = np.asarray(im)
                vals, counts = np.unique(arr, return_counts=True)

                mask_pixel_rows.append({'category':category,'defect_type':defect,'file':p.name,
                    'shape':str(arr.shape),'dtype':str(arr.dtype),
                    'unique_values':json.dumps(vals.tolist()),
                    'value_counts':json.dumps(counts.tolist()),
                    'nonzero_fraction':float(np.count_nonzero(arr)/arr.size)})
            except Exception as e:
                mask_pixel_rows.append({'category':category,'defect_type':defect,'file':p.name,'read_error':repr(e)})
mask_pixels_df = pd.DataFrame(mask_pixel_rows)
mask_pixels_df.to_csv(OUT_DIR/'sampled_mask_pixel_audit.csv', index=False)
display(mask_pixels_df.head(30))
print('Mask pixel arrays decoded:', len(mask_pixels_df))


## 4. Compact preprocessing summary
This writes a concise JSON summary from the full header scans and sampled pixel audits. Inspect the CSVs for details.

In [ ]:
def frequency(series):
    return {str(k):int(v) for k,v in series.value_counts(dropna=False).items()}

handoff = {
  'dataset_root_layout': 'root/category/{train/good, test/defect_type, ground_truth/defect_type}',
  'categories': categories,
  'full_scan_counts': {'images':int(len(images_df)), 'masks':int(len(masks_df)),
                       'missing_anomalous_test_masks':int(len(missing_masks))},
  'image_modes':frequency(images_df['mode']) if 'mode' in images_df else {},
  'image_formats':frequency(images_df['format']) if 'format' in images_df else {},
  'image_dimensions':frequency(images_df['width'].astype(str)+'x'+images_df['height'].astype(str)) if {'width','height'}.issubset(images_df.columns) else {},
  'mask_modes':frequency(masks_df['mode']) if 'mode' in masks_df else {},
  'mask_formats':frequency(masks_df['format']) if 'format' in masks_df else {},
  'mask_dimensions':frequency(masks_df['width'].astype(str)+'x'+masks_df['height'].astype(str)) if {'width','height'}.issubset(masks_df.columns) else {},
  'pixel_sample_count':int(len(pixels_df)),
  'mask_pixel_sample_count':int(len(mask_pixels_df)),
  'pixel_samples_csv':'sampled_image_pixel_statistics.csv',
  'mask_samples_csv':'sampled_mask_pixel_audit.csv',
  'full_image_metadata_csv':'all_image_metadata.csv',
  'full_mask_metadata_csv':'all_mask_metadata.csv',
  'missing_mask_pairs_csv':'missing_mask_pairs.csv',
  'loader_contract': 'Dataset loader returns original NumPy arrays; no resizing, colour conversion, scaling, normalization, thresholding, or tensor conversion.',
  'interpretation_note': 'Header metadata is scanned across all files. Pixel statistics and mask value/coverage are sampled deterministically; do not treat sample statistics as exhaustive.'
}
with open(OUT_DIR/'preprocessing_handoff_fast.json','w') as f:
    json.dump(handoff,f,indent=2)
print(json.dumps(handoff,indent=2))
print('\nSaved all audit outputs to:', OUT_DIR)
